In [1]:
#ERA5 Parse Data
import cdsapi

#https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=download

dataset = "reanalysis-era5-single-levels-timeseries"
request = {
    "variable": [
        "2m_dewpoint_temperature",
        "10m_wind_gust_since_previous_post_processing",
        "skin_temperature",
        "surface_pressure",
        "surface_solar_radiation_downwards",
        "2m_temperature",
        "total_precipitation",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind"
    ],
    "location": {"longitude": 48, "latitude": 20.5},
    "date": ["2026-01-01/2026-05-06"],
    "data_format": "netcdf"
 #   "data_format": "csv"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download("era5_data_fix_live.zip")



2026-05-08 05:22:34,243 INFO [2026-02-16T00:00:00] - To generate this ERA5 hourly time series dataset, **homogenisation conventions have been applied to the ERA5 source GRIB data** to ensure consistency, usability, and alignment across chosen variables and time steps. The processed data were then written to an **ARCO Zarr archive**, enabling efficient cloud-optimised access and scalable data retrieval. Please refer to the [user guide](https://confluence.ecmwf.int/x/R6cfHg) for details.

- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=overview). **Requirements for additional parameters may be considered**. Please raise your request with ECMWF Support [here](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202).
2026-05-08 05:22:34,244 INFO Request ID is eefb5636-6fc9-4ce0-8210-c379b7f4d8bf
2026-05-08 05:22:34,326 INF

af1a69606bfff9971f5bc29701e4ea8f.zip:   0%|          | 0.00/190k [00:00<?, ?B/s]

'era5_data_fix_live.zip'

In [2]:
# 2. NetCDF beolvasása 
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine
import pandas as pd
import shutil
import xarray as xr
import zipfile

load_dotenv() # betölti a .env fájlt

db_host = os.getenv("DB_HOST")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")

engine = create_engine( f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}")

zip_path = "era5_data_fix_live.zip"
extract_dir = "extracted-fix-live"

# 1. ZIP kibontása
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

# 2. Összegyűjtjük az összes .nc fájlt
nc_files = [os.path.join(extract_dir, f) for f in os.listdir(extract_dir) if f.endswith(".nc")]

if not nc_files:
    raise FileNotFoundError("Nem található .nc fájl a ZIP-ben!")

print("Megtalált NetCDF fájlok:")
for f in nc_files:
    print(" -", f)

# 3. Mindegyiket beolvassuk és DataFrame-re alakítjuk
dfs = []
for nc in nc_files:
    print("Beolvasás:", nc)
    ds = xr.open_dataset(nc)
    df = ds.to_dataframe().reset_index()
    dfs.append(df)

# 3. NetCDF-ek összeolvasztása
datasets = [xr.open_dataset(nc) for nc in nc_files]
ds_merged = xr.merge(datasets, compat="override")

# 4. DataFrame konverzió
df = ds_merged.to_dataframe().reset_index()

# --- SZÁMOLT ÉRTÉKEK HOZZÁADÁSA ---

import numpy as np
#Több értéket kénytelen vagyok előfeldolgozni, mert a modellnek logikusabb ebből tanulni mert ilyen értékeken fogja kapni a predikcióhoz az előzményeket.
df["temp_c"] = df["t2m"] - 273.15 #Konvertálás K-ból C-be.
df["dewpoint_c"] = df["d2m"] - 273.15 #Konvertálás K-ból C-be.
df["skt_c"] = df["skt"] - 273.15 #Konvertálás K-ból C-be.
df["sp_hpa"] = df["sp"] / 100 #Itt konvertálom a Pa kapott értéket hpa értékbe.
df["ssrd_wm2"] = df["ssrd"] / 3600 #Így kapunk az egyébként J/m2 ből egy átlagos W/m2-t
df["fg10_kmh"] = df["fg10"] * 3.6 #Itt konvertálom a m/s-ban érkezőt km/h-ra.

#A páratartalomhoz át kell számolni a C és dewpoint értékekből.
def saturation_vapor_pressure(T):
    return 6.112 * np.exp((17.67 * T) / (T + 243.5))

e_t = saturation_vapor_pressure(df["temp_c"])
e_td = saturation_vapor_pressure(df["dewpoint_c"])

df["rh"] = 100 * (e_td / e_t)
df["rh"] = df["rh"].clip(0, 100) # százalékban kell

df["wind_speed"] = np.sqrt(df["u10"]**2 + df["v10"]**2) * 3.6 #A vektorokból kell átszámolni a szélsebességet km/h-ba

df["wind_dir"] = np.degrees(np.arctan2(df["u10"], df["v10"]))
df["wind_dir"] = (df["wind_dir"] + 360) % 360 #A vektorokból kell átszámolni a szélirányt °-ba.

# 5. Típusok normalizálása
df = df.convert_dtypes()
df = df.where(df.notnull(), None)

# --- OSZLOPOK ELTÁVOLÍTÁSA ---
# Itt sorold fel azokat az eredeti oszlopokat (pl. t2m, d2m, sp, u10, v10 stb.),
# amikre már nincs szükséged, mert kiszámoltad belőlük a végleges verziót.
cols_to_drop = ["t2m", "d2m", "skt", "sp", "ssrd", "fg10", "u10", "v10"]

# Csak azokat próbáljuk törölni, amik tényleg benne vannak a df-ben (biztonsági okból)
existing_cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=existing_cols_to_drop)

# 6. Betöltés MySQL-be
df.to_sql("weather_live", engine, if_exists="replace", index=False, chunksize=5000)

print("Sikeresen betöltve!")


Megtalált NetCDF fájlok:
 - extracted-fix-live/reanalysis-era5-single-levels-timeseries-sfcn3imcsq6.nc
Beolvasás: extracted-fix-live/reanalysis-era5-single-levels-timeseries-sfcn3imcsq6.nc
Sikeresen betöltve!
